In [ ]:
from google.colab import drive
import os, subprocess, sys, re, time, threading, requests, shutil

drive.mount('/content/drive', force_remount=True)

DRIVE_BASE   = '/content/drive/MyDrive/RemovationAI'
DRIVE_MODELS = f'{DRIVE_BASE}/models'
DRIVE_NODES  = f'{DRIVE_BASE}/custom_nodes'
DRIVE_COMFY  = f'{DRIVE_BASE}/ComfyUI'
COMFYUI_PATH = '/content/ComfyUI'
NODES_PATH   = f'{COMFYUI_PATH}/custom_nodes'

for d in [DRIVE_BASE, DRIVE_MODELS, DRIVE_NODES]:
    os.makedirs(d, exist_ok=True)

MODEL_SUBDIRS = [
    'checkpoints','controlnet','loras','vae','clip','clip_vision',
    'upscale_models','unet','diffusion_models','style_models',
    'inpaint','sam2','depth_anything_v2','IC-Light',
    'grounding-dino','insightface','onnx','ipadapter','sams'
]
for s in MODEL_SUBDIRS:
    os.makedirs(f'{DRIVE_MODELS}/{s}', exist_ok=True)

print('✅ Drive mounted')
print(f'   Storage: {DRIVE_BASE}')
print(f'   First run: ~20 min  |  After that: ~2-3 min')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted
   Storage: /content/drive/MyDrive/RemovationAI
   First run: ~20 min  |  After that: ~2-3 min


In [ ]:
import os, subprocess, sys, re, time, threading, requests, shutil

DRIVE_BASE   = '/content/drive/MyDrive/RemovationAI'
DRIVE_MODELS = f'{DRIVE_BASE}/models'
DRIVE_NODES  = f'{DRIVE_BASE}/custom_nodes'
DRIVE_COMFY  = f'{DRIVE_BASE}/ComfyUI'
COMFYUI_PATH = '/content/ComfyUI'
NODES_PATH   = f'{COMFYUI_PATH}/custom_nodes'

MODEL_SUBDIRS = [
    'checkpoints','controlnet','loras','vae','clip','clip_vision',
    'upscale_models','unet','diffusion_models','style_models',
    'inpaint','sam2','depth_anything_v2','IC-Light',
    'grounding-dino','insightface','onnx','ipadapter','sams',
]

if shutil.which('nvidia-smi') is None:
    print('❌ No GPU runtime — go to: Runtime → Change runtime type → T4 GPU → Save')
    print('   Then: Runtime → Disconnect and delete runtime → Run all')
    sys.exit(1)

result = subprocess.run(
    ['nvidia-smi','--query-gpu=name,memory.total,memory.free','--format=csv,noheader,nounits'],
    capture_output=True, text=True
)
if result.returncode == 0:
    name, total, free = [x.strip() for x in result.stdout.strip().split(',')]
    print(f'✅ GPU: {name}')
    print(f'   VRAM: {int(total)/1024:.1f} GB total, {int(free)/1024:.1f} GB free')
    if int(total) < 10000:
        print('⚠️  Less than 10GB VRAM — large models may OOM')
else:
    print('❌ GPU check failed — change runtime to T4 GPU')
    sys.exit(1)


FileNotFoundError: [Errno 2] No such file or directory: 'nvidia-smi'

In [ ]:
import os, subprocess, sys, re, time, threading, requests, shutil

DRIVE_BASE   = '/content/drive/MyDrive/RemovationAI'
DRIVE_MODELS = f'{DRIVE_BASE}/models'
DRIVE_NODES  = f'{DRIVE_BASE}/custom_nodes'
DRIVE_COMFY  = f'{DRIVE_BASE}/ComfyUI'
COMFYUI_PATH = '/content/ComfyUI'
NODES_PATH   = f'{COMFYUI_PATH}/custom_nodes'

MODEL_SUBDIRS = [
    'checkpoints','controlnet','loras','vae','clip','clip_vision',
    'upscale_models','unet','diffusion_models','style_models',
    'inpaint','sam2','depth_anything_v2','IC-Light',
    'grounding-dino','insightface','onnx','ipadapter','sams',
]

if os.path.exists(f'{DRIVE_COMFY}/main.py'):
    print('ComfyUI cached on Drive — copying to /content...')
    if os.path.exists(COMFYUI_PATH):
        os.system(f'rm -rf {COMFYUI_PATH}')
    os.system(f'cp -r {DRIVE_COMFY} {COMFYUI_PATH}')
    print('✅ ComfyUI ready from cache')
else:
    print('First time — cloning ComfyUI (~2 min)...')
    os.system(f'git clone https://github.com/comfyanonymous/ComfyUI.git {DRIVE_COMFY} -q')
    os.system(f'cp -r {DRIVE_COMFY} {COMFYUI_PATH}')
    print('✅ ComfyUI cloned and cached')

os.system(f'pip install -q -r {COMFYUI_PATH}/requirements.txt')
os.system('pip install -q xformers --index-url https://download.pytorch.org/whl/cu121 2>/dev/null || true')
os.system('pip install -q triton 2>/dev/null || true')

# Symlink all model dirs from Drive to ComfyUI
comfy_models = f'{COMFYUI_PATH}/models'
os.makedirs(comfy_models, exist_ok=True)
for subdir in MODEL_SUBDIRS:
    src  = f'{DRIVE_MODELS}/{subdir}'
    link = f'{comfy_models}/{subdir}'
    os.makedirs(src, exist_ok=True)
    if os.path.islink(link) or os.path.exists(link):
        if os.path.islink(link): pass
        else: continue
    else:
        os.symlink(src, link)

print('✅ All model dirs linked from Drive (permanent storage)')

In [ ]:
import os, subprocess, sys, re, time, threading, requests, shutil

DRIVE_BASE   = '/content/drive/MyDrive/RemovationAI'
DRIVE_MODELS = f'{DRIVE_BASE}/models'
DRIVE_NODES  = f'{DRIVE_BASE}/custom_nodes'
DRIVE_COMFY  = f'{DRIVE_BASE}/ComfyUI'
COMFYUI_PATH = '/content/ComfyUI'
NODES_PATH   = f'{COMFYUI_PATH}/custom_nodes'

MODEL_SUBDIRS = [
    'checkpoints','controlnet','loras','vae','clip','clip_vision',
    'upscale_models','unet','diffusion_models','style_models',
    'inpaint','sam2','depth_anything_v2','IC-Light',
    'grounding-dino','insightface','onnx','ipadapter','sams',
]

sam2_src   = f'{DRIVE_MODELS}/sam2/sam2_hiera_small.pt'
gdino_src  = f'{DRIVE_MODELS}/grounding-dino/groundingdino_swint_ogc.pth'
depth_src  = f'{DRIVE_MODELS}/depth_anything_v2/depth_anything_v2_vits.pth'


# 1. SAM2 — Impact Pack looks in models/sams/ (not models/sam2/)
sams_dir = f'{COMFYUI_PATH}/models/sams'
os.makedirs(sams_dir, exist_ok=True)
for t in [
    f'{sams_dir}/sam2_hiera_small.pt',
    f'{NODES_PATH}/comfyui-impact-pack/models/sam2/sam2_hiera_small.pt',
    f'{NODES_PATH}/comfyui-segment-anything/models/sam2/sam2_hiera_small.pt',
]:
    os.makedirs(os.path.dirname(t), exist_ok=True)
    if not os.path.exists(t) and os.path.exists(sam2_src):
        shutil.copy2(sam2_src, t)
print('✅ SAM2 paths fixed')

# 2. GroundingDINO — Impact Pack and segment-anything each need their own copy
for t in [
    f'{NODES_PATH}/comfyui-segment-anything/models/groundingdino/groundingdino_swint_ogc.pth',
    f'{NODES_PATH}/comfyui-impact-pack/models/grounding-dino/groundingdino_swint_ogc.pth',
]:
    os.makedirs(os.path.dirname(t), exist_ok=True)
    if not os.path.exists(t) and os.path.exists(gdino_src):
        shutil.copy2(gdino_src, t)
print('✅ GroundingDINO paths fixed')

# 3. Depth Anything — controlnet_aux looks in its own ckpts folder
depth_target = f'{NODES_PATH}/comfyui_controlnet_aux/ckpts/depth_anything_v2_vits.pth'
os.makedirs(os.path.dirname(depth_target), exist_ok=True)
if not os.path.exists(depth_target) and os.path.exists(depth_src):
    shutil.copy2(depth_src, depth_target)
print('✅ Depth Anything path fixed')

# 4. CLIPSeg — must be cloned directly into ComfyUI (not Drive symlink)
#    Impact Pack checks: "CLIPSeg" in nodes.NODE_CLASS_MAPPINGS
clipseg_path = f'{NODES_PATH}/ComfyUI-CLIPSeg'
if not os.path.exists(clipseg_path):
    print('⏬ Installing CLIPSeg (required for Room Changer)...')
    ret = os.system(f'git clone https://github.com/time-river/ComfyUI-CLIPSeg.git {clipseg_path} -q')
    if ret == 0:
        req = f'{clipseg_path}/requirements.txt'
        if os.path.exists(req):
            os.system(f'pip install -q -r {req} 2>/dev/null || true')
        os.system('pip install -q transformers')
        print('✅ CLIPSeg installed')
    else:
        print('❌ CLIPSeg clone failed — Room Changer may not work')
else:
    print('✅ CLIPSeg already installed')

# 5. LaMa — needs to be in the custom node's own models folder
lama_node_models = f'{NODES_PATH}/comfyui-lama-remover/models'
lama_drive       = f'{DRIVE_NODES}/comfyui-lama-remover/models/big-lama.pt'
lama_target      = f'{lama_node_models}/big-lama.pt'
os.makedirs(lama_node_models, exist_ok=True)
if not os.path.exists(lama_target):
    if os.path.exists(lama_drive):
        shutil.copy2(lama_drive, lama_target)
        print('✅ LaMa model copied from Drive')
    else:
        print('⏬ Downloading LaMa model...', end=' ', flush=True)
        os.system(f'wget -q -c -O "{lama_drive}" "https://huggingface.co/smartywu/big-lama/resolve/main/big-lama.pt"')
        if os.path.exists(lama_drive):
            shutil.copy2(lama_drive, lama_target)
            print('✅')
        else:
            print('❌')
else:
    print('✅ LaMa model ready')

print('\n✅ All model paths fixed')

In [ ]:
import os, subprocess, sys, re, time, threading, requests, shutil

DRIVE_BASE   = '/content/drive/MyDrive/RemovationAI'
DRIVE_MODELS = f'{DRIVE_BASE}/models'
DRIVE_NODES  = f'{DRIVE_BASE}/custom_nodes'
DRIVE_COMFY  = f'{DRIVE_BASE}/ComfyUI'
COMFYUI_PATH = '/content/ComfyUI'
NODES_PATH   = f'{COMFYUI_PATH}/custom_nodes'

MODEL_SUBDIRS = [
    'checkpoints','controlnet','loras','vae','clip','clip_vision',
    'upscale_models','unet','diffusion_models','style_models',
    'inpaint','sam2','depth_anything_v2','IC-Light',
    'grounding-dino','insightface','onnx','ipadapter','sams',
]

# Essential pip packages for all features
print('Installing core dependencies...')
os.system('pip install -q torchvision torchaudio')
os.system('pip install -q transformers accelerate diffusers safetensors')
os.system('pip install -q timm einops kornia')
os.system('pip install -q groundingdino-py 2>/dev/null || true')  # fallback
os.system('pip install -q supervision ultralytics 2>/dev/null || true')
print('✅ Core packages installed')

NODES_PATH = f'{COMFYUI_PATH}/custom_nodes'
os.makedirs(NODES_PATH, exist_ok=True)

REQUIRED_NODES = [
    ('ComfyUI-Manager',             'https://github.com/ltdrdata/ComfyUI-Manager.git'),
    ('comfyui_controlnet_aux',      'https://github.com/Fannovel16/comfyui_controlnet_aux.git'),
    ('ComfyUI-Advanced-ControlNet', 'https://github.com/Kosinkadink/ComfyUI-Advanced-ControlNet.git'),
    ('comfyui-inpaint-nodes',       'https://github.com/Acly/comfyui-inpaint-nodes.git'),
    ('comfyui-lama-remover',        'https://github.com/Layer-norm/comfyui-lama-remover.git'),
    ('ComfyUI-IC-Light',            'https://github.com/huchenlei/ComfyUI-IC-Light.git'),
    ('comfyui-impact-pack',         'https://github.com/ltdrdata/ComfyUI-Impact-Pack.git'),
    ('rgthree-comfy',               'https://github.com/rgthree/rgthree-comfy.git'),
    ('comfyui-kjnodes',             'https://github.com/kijai/ComfyUI-KJNodes.git'),
    ('ComfyUI_IPAdapter_plus',      'https://github.com/cubiq/ComfyUI_IPAdapter_plus.git'),
    ('was-node-suite-comfyui',      'https://github.com/WASasquatch/was-node-suite-comfyui.git'),
    ('ComfyUI-GGUF',                'https://github.com/city96/ComfyUI-GGUF.git'),
    ('ComfyUI-RMBG',                'https://github.com/1038lab/ComfyUI-RMBG.git'),
    ('comfyui-segment-anything',    'https://github.com/storyicon/comfyui_segment_anything.git'),
    ('ComfyUI-FluxKontext',         'https://github.com/lum3on/ComfyUI-FluxKontext.git'),
    ('comfyui-ollama',              'https://github.com/stavsap/comfyui-ollama.git'),
    ('ComfyUI-VideoHelperSuite',    'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git'),
]

for name, url in REQUIRED_NODES:
    drive_node = f'{DRIVE_NODES}/{name}'
    comfy_node = f'{NODES_PATH}/{name}'
    if not os.path.exists(comfy_node):
        if os.path.exists(drive_node):
            if not os.path.islink(comfy_node):
                os.symlink(drive_node, comfy_node)
            print(f'🔗 {name} (Drive cache)')
        else:
            print(f'⏬ {name}...')
            ret = os.system(f'git clone {url} {drive_node} -q')
            if ret == 0 and not os.path.islink(comfy_node):
                os.symlink(drive_node, comfy_node)
                print(f'✅ {name}')
            else:
                print(f'❌ {name} failed')
    else:
        print(f'✅ {name}')
        
    # Always install requirements if they exist to support fresh runtime environments
    req = f'{comfy_node}/requirements.txt'
    if os.path.exists(req):
        os.system(f'pip install -q -r {req} 2>/dev/null || true')

# 4. CLIPSeg — must be cloned directly into ComfyUI (not Drive symlink)
#    Impact Pack checks: "CLIPSeg" in nodes.NODE_CLASS_MAPPINGS
clipseg_path = f'{NODES_PATH}/ComfyUI-CLIPSeg'
if not os.path.exists(clipseg_path):
    print('⏬ Installing CLIPSeg (required for Room Changer)...')
    ret = os.system(f'git clone https://github.com/time-river/ComfyUI-CLIPSeg.git {clipseg_path} -q')
    if ret == 0:
        print('✅ CLIPSeg installed')
    else:
        print('❌ CLIPSeg clone failed — Room Changer may not work')
else:
    print('✅ CLIPSeg ready')

# Always install CLIPSeg requirements if they exist
clipseg_req = f'{clipseg_path}/requirements.txt'
if os.path.exists(clipseg_req):
    os.system(f'pip install -q -r {clipseg_req} 2>/dev/null || true')

os.system('cd /content/ComfyUI/custom_nodes/comfyui-impact-pack && python install.py 2>/dev/null || true')
print('✅ All custom nodes ready')


In [ ]:
import os, subprocess, sys, re, time, threading, requests, shutil

DRIVE_BASE   = '/content/drive/MyDrive/RemovationAI'
DRIVE_MODELS = f'{DRIVE_BASE}/models'
DRIVE_NODES  = f'{DRIVE_BASE}/custom_nodes'
DRIVE_COMFY  = f'{DRIVE_BASE}/ComfyUI'
COMFYUI_PATH = '/content/ComfyUI'
NODES_PATH   = f'{COMFYUI_PATH}/custom_nodes'

MODEL_SUBDIRS = [
    'checkpoints','controlnet','loras','vae','clip','clip_vision',
    'upscale_models','unet','diffusion_models','style_models',
    'inpaint','sam2','depth_anything_v2','IC-Light',
    'grounding-dino','insightface','onnx','ipadapter','sams',
]

def get_model(subdir, filename, url, label, gated=False):
    drive_path = f'{DRIVE_MODELS}/{subdir}/{filename}'
    comfy_path = f'{COMFYUI_PATH}/models/{subdir}/{filename}'
    if os.path.exists(comfy_path) and not os.path.islink(comfy_path):
        print(f'✅ {label}')
        return True
    if os.path.exists(drive_path) and os.path.getsize(drive_path) > 100_000:
        print(f'✅ {label} (Drive)')
        return True
    if gated:
        print(f'⚠️  UPLOAD NEEDED: {label}')
        print(f'   → {drive_path}')
        return False
    print(f'⏬ {label}...', end=' ', flush=True)
    os.makedirs(os.path.dirname(drive_path), exist_ok=True)
    ret = os.system(f'wget -q -c -O "{drive_path}" "{url}"')
    if os.path.exists(drive_path) and os.path.getsize(drive_path) > 100_000:
        print('✅')
        return True
    print('❌ failed')
    return False

print('=== MODELS (downloaded once to Drive) ===\n')

# Checkpoints
get_model('checkpoints','sd_xl_base_1.0.safetensors',
    'https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors',
    'SDXL Base 1.0 [Room Changer / Redesign / Ceiling / Ultra-Pro]')
get_model('checkpoints','photon_v1.safetensors',
    'https://huggingface.co/digiplay/Photon_v1/resolve/main/photon_v1.safetensors',
    'photon_v1 [Style Transfer base]')
get_model('checkpoints','1_5photon_v1.safetensors',
    'https://huggingface.co/digiplay/Photon_v1/resolve/main/photon_v1.safetensors',
    '1_5photon_v1 [IC-Light base]')
get_model('checkpoints','juggernautXL_version6Rundiffusion.safetensors',
    'https://huggingface.co/RunDiffusion/Juggernaut-XL-v6/resolve/main/juggernautXL_version6Rundiffusion.safetensors',
    'JuggernautXL v6 [Outpaint]')

# ControlNet
get_model('controlnet','control_v11p_sd15_canny.pth',
    'https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_canny.pth',
    'ControlNet Canny SD1.5 [Redesign]')
get_model('controlnet','control_v11p_sd15_mlsd_fp16.safetensors',
    'https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_mlsd.pth',
    'ControlNet MLSD SD1.5 [Style Transfer]')
get_model('controlnet','controlnet-canny-sdxl.safetensors',
    'https://huggingface.co/xinsir/controlnet-canny-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors',
    'ControlNet Canny SDXL [Ultra-Pro]')

# Upscale
get_model('upscale_models','RealESRGAN_x4plus.pth',
    'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
    'RealESRGAN x4plus [Upscale]')
get_model('upscale_models','RealESRGAN_x4.pth',
    'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth',
    'RealESRGAN x4 general [Upscale]')


# SDXL VAE (needed by Room Changer, Object Removal, Redesign, Ultra-Pro)
get_model('vae','sdxl_vae.safetensors',
    'https://huggingface.co/madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl_vae.safetensors',
    'SDXL VAE fp16-fix [Room Changer / Object Removal / Ultra-Pro]')

# RealVisXL (needed by Room Changer color recolor)
get_model('checkpoints','realvisxlV40.safetensors',
    'https://huggingface.co/SG161222/RealVisXL_V4.0/resolve/main/RealVisXL_V4.0.safetensors',
    'RealVisXL V4.0 [Room Changer Recolor]')

# Control-LoRA SDXL rank-256 (needed by Ultra-Pro and Redesign stages)
get_model('controlnet','control-lora-canny-rank256.safetensors',
    'https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-canny-rank256.safetensors',
    'Control-LoRA Canny rank256 [Ultra-Pro / Redesign]')
get_model('controlnet','control-lora-depth-rank256.safetensors',
    'https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-depth-rank256.safetensors',
    'Control-LoRA Depth rank256 [Ultra-Pro]')
get_model('controlnet','control-lora-sketch-rank256.safetensors',
    'https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-sketch-rank256.safetensors',
    'Control-LoRA Sketch rank256 [Ultra-Pro]')
get_model('controlnet','control-lora-recolor-rank256.safetensors',
    'https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-recolor-rank256.safetensors',
    'Control-LoRA Recolor rank256 [Room Changer Color]')

# IPAdapter SDXL (needed by Furniture Placement / Redesign stage-3)
os.makedirs(f'{DRIVE_MODELS}/ipadapter', exist_ok=True)
get_model('ipadapter','ip-adapter-plus_sdxl_vit-h.safetensors',
    'https://huggingface.co/h94/IP-Adapter/resolve/main/sdxl_models/ip-adapter-plus_sdxl_vit-h.safetensors',
    'IP-Adapter Plus SDXL ViT-H [Furniture Placement / Redesign]')

# CLIP Vision for IPAdapter (must be named exactly)
get_model('clip_vision','CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors',
    'https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors',
    'CLIP-ViT-H-14 [IPAdapter companion]')

# IC-Light (fixed URL)
get_model('IC-Light','iclight_sd15_fc.safetensors',
    'https://huggingface.co/huchenlei/IC-Light/resolve/main/iclight_sd15_fc.safetensors',
    'IC-Light fc [Relighting]')

# SAM2 + Depth + GroundingDINO
get_model('sam2','sam2_hiera_small.pt',
    'https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt',
    'SAM2 Hiera Small [Scene Understanding]')
get_model('depth_anything_v2','depth_anything_v2_vits.pth',
    'https://huggingface.co/depth-anything/Depth-Anything-V2-Small/resolve/main/depth_anything_v2_vits.pth',
    'Depth Anything V2 [Ultra-Pro]')
get_model('grounding-dino','groundingdino_swint_ogc.pth',
    'https://huggingface.co/ShilongLiu/GroundingDINO/resolve/main/groundingdino_swint_ogc.pth',
    'GroundingDINO [Furniture Removal / Ultra-Pro]')

# Inpaint (fixed URL)
get_model('inpaint','fooocus_inpaint_head.pth',
    'https://huggingface.co/lllyasviel/fooocus_inpaint/resolve/main/fooocus_inpaint_head.pth',
    'Fooocus Inpaint Head [Outpaint]')

# Flux encoders
get_model('clip','clip_l.safetensors',
    'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors',
    'CLIP-L [Flux Staging]')
get_model('clip','t5xxl_fp8_e4m3fn.safetensors',
    'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors',
    'T5-XXL fp8 [Flux Staging]')
get_model('clip_vision','sigclip_vision_patch14_384.safetensors',
    'https://huggingface.co/Comfy-Org/sigclip_vision_384/resolve/main/sigclip_vision_patch14_384.safetensors',
    'SigCLIP Vision [Flux Staging]')

# Gated models check
print('\n=== GATED MODELS (upload to Drive manually — one time only) ===')
GATED = [
    ('diffusion_models','flux1-dev-fp8.safetensors',              'Furniture Staging',         'https://huggingface.co/black-forest-labs/FLUX.1-dev'),
    ('vae',             'ae.safetensors',                         'Furniture Staging VAE',      'https://huggingface.co/black-forest-labs/FLUX.1-dev'),
    ('style_models',    'flux1-redux-dev.safetensors',            'Furniture Staging Redux',    'https://huggingface.co/black-forest-labs/FLUX.1-Redux-dev'),
    ('loras',           'flux_realism_lora.safetensors',          'Furniture Staging LoRA',     'https://huggingface.co/XLabs-AI/flux-RealismLora'),
    ('diffusion_models','qwen_image_edit_2509_fp8_e4m3fn.safetensors','Material Changer',       'your local copy'),
    ('clip',            'qwen_2.5_vl_7b_fp8_scaled.safetensors', 'Material Changer CLIP',      'your local copy'),
    ('vae',             'qwen_image_vae.safetensors',             'Material Changer VAE',       'your local copy'),
    ('loras',           'Qwen-Edit-2509-Multiple-angles.safetensors','Material Changer LoRA',   'your local copy'),
    ('unet',            'qwen-image-edit-2509.Q4_K_M.gguf',      'Material Changer GGUF',      'your local copy'),
]
missing_count = 0
for subdir, fname, feature, url in GATED:
    drive_path = f'{DRIVE_MODELS}/{subdir}/{fname}'
    found = os.path.exists(drive_path) and os.path.getsize(drive_path) > 100_000
    if found:
        print(f'  ✅ {fname}')
    else:
        missing_count += 1
        print(f'  ❌ {fname} — {feature}')

if missing_count:
    print(f'\n  Upload {missing_count} model(s) to Drive to unlock those features')
    print(f'  Other features work without them')
else:
    print('\n  ✅ All gated models present — full feature set available')

In [ ]:
import os, subprocess, sys, re, time, threading, requests, shutil

DRIVE_BASE   = '/content/drive/MyDrive/RemovationAI'
DRIVE_MODELS = f'{DRIVE_BASE}/models'
DRIVE_NODES  = f'{DRIVE_BASE}/custom_nodes'
DRIVE_COMFY  = f'{DRIVE_BASE}/ComfyUI'
COMFYUI_PATH = '/content/ComfyUI'
NODES_PATH   = f'{COMFYUI_PATH}/custom_nodes'

MODEL_SUBDIRS = [
    'checkpoints','controlnet','loras','vae','clip','clip_vision',
    'upscale_models','unet','diffusion_models','style_models',
    'inpaint','sam2','depth_anything_v2','IC-Light',
    'grounding-dino','insightface','onnx','ipadapter','sams',
]

if not os.path.exists('/content/cloudflared'):
    os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared')

comfy_log = open('/content/comfyui.log', 'w')
comfy_proc = subprocess.Popen(
    ['python','main.py','--listen','0.0.0.0','--port','8188','--preview-method','none'],
    cwd=COMFYUI_PATH, stdout=comfy_log, stderr=comfy_log
)

# Extended timeout — RMBG alone takes 42s to load, total ~3 min
print('Starting ComfyUI (up to 4.5 min for all nodes to load)', end='', flush=True)
comfy_ok = False
for _ in range(90):
    time.sleep(3)
    print('.', end='', flush=True)
    try:
        if requests.get('http://localhost:8188/system_stats', timeout=2).status_code == 200:
            print(' ✅')
            comfy_ok = True
            break
    except: pass
else:
    print(' ❌')
    print('----- LAST 100 LINES OF COMFYUI.LOG -----')
    try:
        with open('/content/comfyui.log') as f:
            lines = f.readlines()
            print(''.join(lines[-100:]))
    except Exception as le:
        print(f'Could not read log file: {le}')

tunnel_log = open('/content/tunnel.log', 'w')
subprocess.Popen(
    ['/content/cloudflared','tunnel','--url','http://localhost:8188'],
    stdout=tunnel_log, stderr=tunnel_log
)

print('Starting tunnel', end='', flush=True)
tunnel_url = None
for _ in range(30):
    time.sleep(2)
    print('.', end='', flush=True)
    try:
        with open('/content/tunnel.log') as f:
            m = re.search(r'https://[\w-]+\.trycloudflare\.com', f.read())
        if m:
            tunnel_url = m.group(0)
            print(' ✅')
            break
    except: pass
else:
    print(' ❌  check: !cat /content/tunnel.log')

if tunnel_url:
    print(f'''
+------------------------------------------------------+
|           REMOVATION.AI COLAB READY                  |
+------------------------------------------------------+
|  1. Copy this URL:                                   |
|     {tunnel_url:<50} |
|                                                      |
|  2. Admin panel -> Settings -> GPU -> Cloud URL      |
|     Paste URL -> Mode: cloud -> Save                 |
|                                                      |
|  3. Submit any room redesign to test                 |
+------------------------------------------------------+
''')


In [ ]:
import os, subprocess, sys, re, time, threading, requests, shutil

DRIVE_BASE   = '/content/drive/MyDrive/RemovationAI'
DRIVE_MODELS = f'{DRIVE_BASE}/models'
DRIVE_NODES  = f'{DRIVE_BASE}/custom_nodes'
DRIVE_COMFY  = f'{DRIVE_BASE}/ComfyUI'
COMFYUI_PATH = '/content/ComfyUI'
NODES_PATH   = f'{COMFYUI_PATH}/custom_nodes'

MODEL_SUBDIRS = [
    'checkpoints','controlnet','loras','vae','clip','clip_vision',
    'upscale_models','unet','diffusion_models','style_models',
    'inpaint','sam2','depth_anything_v2','IC-Light',
    'grounding-dino','insightface','onnx','ipadapter','sams',
]

tunnel_url = globals().get('tunnel_url', None)
if tunnel_url:
    print(f'Verifying connection to tunnel: {tunnel_url}...')
    for attempt in range(6):
        try:
            resp = requests.get(f'{tunnel_url}/system_stats', timeout=15)
            if resp.status_code == 200:
                d = resp.json()
                dev = d.get('devices', [{}])[0]
                total = dev.get('vram_total', 0) / 1024**3
                free  = dev.get('vram_free',  0) / 1024**3
                print(f'✅ Connected!')
                print(f'   VRAM: {free:.1f} GB free / {total:.1f} GB total')
                print(f'   Ready to accept jobs')
                break
        except Exception as e:
            if attempt < 5:
                print(f'   Attempt {attempt + 1}/6 failed (DNS/tunnel propagating). Retrying in 5 seconds...', flush=True)
                time.sleep(5)
            else:
                print(f'❌ {e}')
                print('   Failed to connect to tunnel. If you just restarted, try re-running this cell in 30 seconds.')
else:
    print('❌ No active tunnel URL found. Run the tunnel setup cell first.')


In [ ]:
import os, subprocess, sys, re, time, threading, requests, shutil

DRIVE_BASE   = '/content/drive/MyDrive/RemovationAI'
DRIVE_MODELS = f'{DRIVE_BASE}/models'
DRIVE_NODES  = f'{DRIVE_BASE}/custom_nodes'
DRIVE_COMFY  = f'{DRIVE_BASE}/ComfyUI'
COMFYUI_PATH = '/content/ComfyUI'
NODES_PATH   = f'{COMFYUI_PATH}/custom_nodes'

MODEL_SUBDIRS = [
    'checkpoints','controlnet','loras','vae','clip','clip_vision',
    'upscale_models','unet','diffusion_models','style_models',
    'inpaint','sam2','depth_anything_v2','IC-Light',
    'grounding-dino','insightface','onnx','ipadapter','sams',
]

def _ping():
    while True:
        try: requests.get('http://localhost:8188/system_stats', timeout=5)
        except: pass
        time.sleep(30)

threading.Thread(target=_ping, daemon=True).start()
print('✅ Keep-alive running (pings every 30s)')
print('   Leave this tab open — Colab stays active')

In [ ]:
import os, subprocess, sys, re, time, threading, requests, shutil

DRIVE_BASE   = '/content/drive/MyDrive/RemovationAI'
DRIVE_MODELS = f'{DRIVE_BASE}/models'
DRIVE_NODES  = f'{DRIVE_BASE}/custom_nodes'
DRIVE_COMFY  = f'{DRIVE_BASE}/ComfyUI'
COMFYUI_PATH = '/content/ComfyUI'
NODES_PATH   = f'{COMFYUI_PATH}/custom_nodes'

MODEL_SUBDIRS = [
    'checkpoints','controlnet','loras','vae','clip','clip_vision',
    'upscale_models','unet','diffusion_models','style_models',
    'inpaint','sam2','depth_anything_v2','IC-Light',
    'grounding-dino','insightface','onnx','ipadapter','sams',
]

# Run this cell anytime the tunnel URL expires or ComfyUI stops
# Takes ~3 min if ComfyUI needs restart, ~30 sec if still running

comfy_ok = False
try:
    requests.get('http://localhost:8188/system_stats', timeout=5)
    comfy_ok = True
    print('✅ ComfyUI still running')
except:
    print('ComfyUI stopped — restarting...')
    os.system('pkill -f "python main.py" 2>/dev/null')
    time.sleep(3)
    comfy_log = open('/content/comfyui.log', 'w')
    subprocess.Popen(
        ['python','main.py','--listen','0.0.0.0','--port','8188','--preview-method','none'],
        cwd=COMFYUI_PATH, stdout=comfy_log, stderr=comfy_log
    )
    print('Waiting', end='', flush=True)
    for _ in range(90):
        time.sleep(3)
        print('.', end='', flush=True)
        try:
            requests.get('http://localhost:8188/system_stats', timeout=2)
            comfy_ok = True
            print(' ✅')
            break
        except: pass
    else:
        print(' ❌')
        print('----- LAST 100 LINES OF COMFYUI.LOG -----')
        try:
            with open('/content/comfyui.log') as f:
                lines = f.readlines()
                print(''.join(lines[-100:]))
        except Exception as le:
            print(f'Could not read log file: {le}')

# Restart tunnel
if not os.path.exists('/content/cloudflared'):
    os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared')

os.system('pkill -f cloudflared 2>/dev/null')
time.sleep(2)
tunnel_log = open('/content/tunnel.log', 'w')
subprocess.Popen(
    ['/content/cloudflared','tunnel','--url','http://localhost:8188'],
    stdout=tunnel_log, stderr=tunnel_log
)

print('Getting tunnel URL', end='', flush=True)
tunnel_url = None
for _ in range(30):
    time.sleep(2)
    print('.', end='', flush=True)
    try:
        with open('/content/tunnel.log') as f:
            m = re.search(r'https://[\w-]+\.trycloudflare\.com', f.read())
        if m:
            tunnel_url = m.group(0)
            print(' ✅')
            break
    except: pass

if tunnel_url:
    print(f'''
+------------------------------------------------------+
|  NEW TUNNEL URL — paste in admin panel:              |
|  {tunnel_url:<50} |
|  Settings -> GPU -> Cloud URL -> Save                 |
+------------------------------------------------------+
''')


ComfyUI stopped — restarting...
Waiting..........................................................................................Getting tunnel URL... ✅

╔══════════════════════════════════════════════════════╗
║  NEW TUNNEL URL — paste in admin panel:              ║
║  https://dns-clicking-webmasters-honors.trycloudflare.com ║
║  Settings → GPU → Cloud URL → Save                  ║
╚══════════════════════════════════════════════════════╝

